# ML-04 ? Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook completes the Week 3 data contract for the selected Content Refresh / refresh-prioritization lane. It uses the FlyRank warehouse release through DuckDB and Hugging Face, and it develops logic on the mid-panel March 2026 partition rather than the final-month `_sample` table.

## 1. Data contract ? five plain-word answers

1. **What one row means for this lane.** In the March warehouse slice, one raw row means one pseudonymized content item for one pseudonymized client on one `report_date`. For the Content Refresh lane, those raw rows are aggregated to one feature-frame row per `client_hash_id ? content_hash_id` so the team can rank content items for review.
2. **Which warehouse table(s) will be used.** I will use `fact_content_daily_performance`, partitioned by `month=2026-03`, because the Content Refresh lane needs observed search-performance signals over time.
3. **Which time window is used.** I will use the mid-panel month `2026-03`. Honest features are calculated from March 1-15, 2026, and the provisional decline proxy is measured from March 16-31, 2026. The `_sample` table is not used because it is the final month, June 2026, and should be treated as sealed test data.
4. **What will be predicted/ranked.** The proxy is whether a content item loses at least 20% of its GSC impressions from the first half of March to the second half of March. The output is a decision-support ranking of content items that may deserve human refresh/review.
5. **One thing deliberately excluded.** Any second-half March performance field, including the label-derived decline flag, is excluded from the honest feature frame because it is not available at the March 16 decision moment.

## 2. Warehouse setup

This follows the repository's warehouse notebook pattern: DuckDB reads Hugging Face parquet paths directly. The token is read from an environment variable or Colab Secret named `HF_TOKEN`; it is never printed or saved in the notebook.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
from IPython.display import display
from sklearn.metrics import roc_auc_score

MONTH = "2026-03"
DECISION_DATE = "2026-03-16"
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if not HF_TOKEN:
    env_candidates = [Path.cwd() / ".env", *[parent / ".env" for parent in Path.cwd().parents]]
    env_path = next((path for path in env_candidates if path.exists()), None)
    if env_path is not None:
        for line in env_path.read_text(encoding="utf-8").splitlines():
            key, sep, value = line.partition("=")
            if sep and key.strip() == "HF_TOKEN":
                HF_TOKEN = value.strip().strip('"').strip("'")
                break

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "Set a Hugging Face READ token in HF_TOKEN or a Colab Secret named HF_TOKEN. "
        "Do not paste the token into this notebook."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

print(f"Warehouse month selected: {MONTH}")
print("Using table: fact_content_daily_performance")
print("Using _sample table?", False)

Warehouse month selected: 2026-03
Using table: fact_content_daily_performance
Using _sample table? False


## 3. Exactly three verification queries

The next three cells are the only verification queries in this notebook. They check grain, row count/date span, and availability using real warehouse fields.

In [2]:
# Verification Query #1 ? grain
verification_query_1 = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows_at_grain
FROM {FACT_DAILY}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.sql(verification_query_1).df()
print("Verification Query #1 ? duplicate raw fact rows at report_date ? client_hash_id ? content_hash_id")
print("Expected result: 0 rows. If empty, the stated raw grain holds for the March partition.")
display(grain_check)

Verification Query #1 ? duplicate raw fact rows at report_date ? client_hash_id ? content_hash_id
Expected result: 0 rows. If empty, the stated raw grain holds for the March partition.


,report_date,client_hash_id,content_hash_id,rows_at_grain


In [3]:
# Verification Query #2 ? row count and date span
verification_query_2 = f"""
SELECT
    COUNT(*) AS rows_in_march_2026,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {FACT_DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

row_count_and_span = con.sql(verification_query_2).df()
print("Verification Query #2 ? row count and date span for the selected mid-panel month")
display(row_count_and_span)

Verification Query #2 ? row count and date span for the selected mid-panel month


,rows_in_march_2026,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


In [4]:
# Verification Query #3 ? availability with IS TRUE
verification_query_3 = f"""
WITH march_rows AS (
    SELECT *
    FROM {FACT_DAILY}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
)
SELECT
    (SELECT COUNT(*) FROM march_rows) AS rows_before_availability_filter,
    (SELECT COUNT(*) FROM march_rows WHERE gsc_data_available IS TRUE) AS rows_after_gsc_data_available_is_true
"""

availability_check = con.sql(verification_query_3).df()
print("Verification Query #3 ? rows surviving `gsc_data_available IS TRUE`")
display(availability_check)

Verification Query #3 ? rows surviving `gsc_data_available IS TRUE`


,rows_before_availability_filter,rows_after_gsc_data_available_is_true
0,9841378,3611061


## 4. Five-feature frame

The honest feature frame uses at most five features, all calculated from March 1-15, 2026. The label/proxy is calculated from March 16-31, 2026 and is not included as a feature.

`impressions_first_half` ? knowable at the decision moment because it sums GSC impressions observed before March 16, 2026.

`clicks_first_half` ? knowable at the decision moment because it sums GSC clicks observed before March 16, 2026.

`avg_position_first_half` ? knowable at the decision moment because it averages GSC position only before March 16, 2026.

`active_days_first_half` ? knowable at the decision moment because it counts pre-decision days with observed impressions.

`ctr_first_half` ? knowable at the decision moment because it is calculated only from first-half clicks and impressions.

In [5]:
feature_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {FACT_DAILY}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE
),
per_content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date < DATE '{DECISION_DATE}' THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN report_date < DATE '{DECISION_DATE}' THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_first_half,
        AVG(CASE WHEN report_date < DATE '{DECISION_DATE}' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_first_half,
        COUNT(DISTINCT CASE WHEN report_date < DATE '{DECISION_DATE}' AND COALESCE(gsc_impressions, 0) > 0 THEN report_date END) AS active_days_first_half,
        SUM(CASE WHEN report_date >= DATE '{DECISION_DATE}' THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_second_half
    FROM march
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    impressions_first_half,
    clicks_first_half,
    avg_position_first_half,
    active_days_first_half,
    CASE
        WHEN impressions_first_half > 0
        THEN 100.0 * clicks_first_half / impressions_first_half
        ELSE NULL
    END AS ctr_first_half,
    impressions_second_half,
    CASE
        WHEN impressions_first_half >= 100
         AND impressions_second_half < 0.80 * impressions_first_half
        THEN 1 ELSE 0
    END AS declined_second_half
FROM per_content
WHERE impressions_first_half >= 100
"""

feature_frame = con.sql(feature_sql).df()
feature_columns = [
    "impressions_first_half",
    "clicks_first_half",
    "avg_position_first_half",
    "active_days_first_half",
    "ctr_first_half",
]

print(f"Feature rows: {len(feature_frame):,}")
print(f"Number of honest features: {len(feature_columns)}")
display(feature_frame[["client_hash_id", "content_hash_id", *feature_columns, "declined_second_half"]].head(10))

Feature rows: 77,540
Number of honest features: 5


,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,active_days_first_half,ctr_first_half,declined_second_half
0,client_e547b89c05043229,content_498a9afb9da07372,1103.0,0.0,5.207792,13,0.000000,1
1,client_e547b89c05043229,content_32185762106c6e3e,1803.0,8.0,4.946986,13,0.443705,0
2,client_e547b89c05043229,content_98279f3863475491,542.0,1.0,17.729633,13,0.184502,0
3,client_e547b89c05043229,content_6b99af0551ff7a09,2730.0,0.0,46.057579,13,0.000000,0
4,client_e547b89c05043229,content_2a9cabd2c0a98312,327.0,0.0,15.387841,13,0.000000,0
5,client_e547b89c05043229,content_e68fa27389ecd5e3,1606.0,1.0,44.312824,13,0.062267,0
6,client_e547b89c05043229,content_fece693bda2a479b,213.0,0.0,15.368847,13,0.000000,0
7,client_e547b89c05043229,content_f27c109f3743223f,721.0,1.0,10.962056,13,0.138696,0
8,client_e547b89c05043229,content_b5c2258907625e39,2088.0,11.0,3.834032,13,0.526820,0
9,client_e547b89c05043229,content_3e6384b733dc7ee6,406.0,0.0,2.861545,13,0.000000,0


## 5. Deliberate label-leakage experiment

This experiment intentionally adds one leaked feature derived from the label/proxy. The honest score is the ROC AUC of a simple transparent priority score built only from first-half information. Then I add `label_derived_decline_signal`, which is just the second-half decline label copied into the feature set. That field is leakage because it is not available on March 16, directly encodes the answer, and would make model performance look trustworthy when it is only memorizing the target.

In [6]:
model_data = feature_frame.dropna(subset=feature_columns + ["declined_second_half"]).copy()

if model_data["declined_second_half"].nunique() < 2:
    raise ValueError("The March feature frame does not contain both label classes after filtering.")

rank_impressions = model_data["impressions_first_half"].rank(pct=True)
rank_clicks = model_data["clicks_first_half"].rank(pct=True)
rank_ctr = model_data["ctr_first_half"].rank(pct=True)
rank_position_opportunity = model_data["avg_position_first_half"].rank(pct=True)

model_data["honest_refresh_priority_score"] = (
    0.35 * rank_impressions
    + 0.25 * rank_clicks
    + 0.20 * rank_ctr
    + 0.20 * rank_position_opportunity
)

honest_score = roc_auc_score(
    model_data["declined_second_half"],
    model_data["honest_refresh_priority_score"]
)

model_data["label_derived_decline_signal"] = model_data["declined_second_half"]
leaked_score = roc_auc_score(
    model_data["declined_second_half"],
    model_data["label_derived_decline_signal"]
)

final_honest_frame = model_data[["client_hash_id", "content_hash_id", *feature_columns, "declined_second_half", "honest_refresh_priority_score"]].copy()
final_honest_score = roc_auc_score(
    final_honest_frame["declined_second_half"],
    final_honest_frame["honest_refresh_priority_score"]
)

leakage_results = pd.DataFrame(
    [
        {"step": "honest score", "score_type": "ROC AUC", "score": honest_score},
        {"step": "with leaked label-derived feature", "score_type": "ROC AUC", "score": leaked_score},
        {"step": "final honest score after removing leakage", "score_type": "ROC AUC", "score": final_honest_score},
    ]
)

print("Leakage experiment: honest score ? leaked score ? honest score after removal")
display(leakage_results)
print("Leaked feature present in final honest frame?", "label_derived_decline_signal" in final_honest_frame.columns)
display(final_honest_frame.head(10))

Leakage experiment: honest score ? leaked score ? honest score after removal

,step,score_type,score
0,honest score,ROC AUC,0.456656
1,with leaked label-derived feature,ROC AUC,1.000000
2,final honest score after removing leakage,ROC AUC,0.456656


Leaked feature present in final honest frame? False


,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,active_days_first_half,ctr_first_half,declined_second_half,honest_refresh_priority_score
0,client_e547b89c05043229,content_498a9afb9da07372,1103.0,0.0,5.207792,13,0.000000,1,0.389500
1,client_e547b89c05043229,content_32185762106c6e3e,1803.0,8.0,4.946986,13,0.443705,0,0.701325
2,client_e547b89c05043229,content_98279f3863475491,542.0,1.0,17.729633,13,0.184502,0,0.554399
3,client_e547b89c05043229,content_6b99af0551ff7a09,2730.0,0.0,46.057579,13,0.000000,0,0.582781
4,client_e547b89c05043229,content_2a9cabd2c0a98312,327.0,0.0,15.387841,13,0.000000,0,0.354803
5,client_e547b89c05043229,content_e68fa27389ecd5e3,1606.0,1.0,44.312824,13,0.062267,0,0.664744
6,client_e547b89c05043229,content_fece693bda2a479b,213.0,0.0,15.368847,13,0.000000,0,0.311665
7,client_e547b89c05043229,content_f27c109f3743223f,721.0,1.0,10.962056,13,0.138696,0,0.542832
8,client_e547b89c05043229,content_b5c2258907625e39,2088.0,11.0,3.834032,13,0.526820,0,0.709697
9,client_e547b89c05043229,content_3e6384b733dc7ee6,406.0,0.0,2.861545,13,0.000000,0,0.250971


The leaked feature cannot be trusted because it is copied from `declined_second_half`, the outcome the score is supposed to rank. At the March 16 decision moment, the second-half March impressions have not happened yet. A suspiciously high or perfect score from that feature would only prove that the answer was smuggled into the input. The final honest score after removing `label_derived_decline_signal` is the one to keep.

## 6. Limitation of this warehouse slice

This notebook uses only the March 2026 partition and filters search rows with `gsc_data_available IS TRUE`. That is appropriate for search-performance refresh framing, but it means the feature frame excludes rows without confirmed GSC availability in March and does not represent clients or content items whose March search data is unavailable in the warehouse slice.

In [7]:
limitation_summary = pd.DataFrame(
    {
        "slice": [MONTH],
        "table": ["fact_content_daily_performance"],
        "availability_filter": ["gsc_data_available IS TRUE"],
        "known_limitation": [
            "Rows without confirmed March GSC availability are excluded from this refresh-prioritization frame."
        ],
    }
)
display(limitation_summary)

,slice,table,availability_filter,known_limitation
0,2026-03,fact_content_daily_performance,gsc_data_available IS TRUE,Rows without confirmed March GSC availability ...


## 7. Self-check

- [x] Five plain-word contract answers are present.
- [x] Exactly three verification queries are present.
- [x] Query outputs are visible after execution.
- [x] Grain is verified using real warehouse fields: `report_date`, `client_hash_id`, and `content_hash_id`.
- [x] Row count and date span are shown.
- [x] Availability uses `IS TRUE`.
- [x] Feature frame contains no more than five features.
- [x] Every feature has an "available when?" explanation.
- [x] One deliberate label-derived leakage experiment is shown.
- [x] The leaked feature is removed before the final honest result.
- [x] One limitation is named.
- [x] Notebook runs from top to bottom without errors with `HF_TOKEN` loaded from the local environment/root `.env`.